# Model Performance Report

This notebook is the display and export surface for the final model-performance table. It uses reusable functions from `src/`, directly displays model performance evidence, and writes one consolidated CSV: `results/model_performance_summary.csv`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import add_visit_time, modelling_pair_count_table
from src.data.trackfa import feature_catalog
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.clinical_validity import clinical_validity
from src.eval.intervals import adjacent_pair_interval_effect_summary, interval_effect_summary, pooled_adjacent_pair_effect_summary
from src.eval.single_feature import single_feature_interval_baselines
from src.eval.stability import selected_feature_jaccard
from src.models.srm_global import srm_global_loocv, srm_global_repeated_group_cv
from src.reporting.model_performance import (
    append_log_model_summaries,
    assemble_performance_rows,
    best_model_rows_from_logs,
    save_one_performance_csv,
)

set_global_seeds(DEFAULT_CONFIG.random_state)
RANDOM_SEED = DEFAULT_CONFIG.random_state
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 300

long_path = REPO_ROOT / "data" / "processed" / "trackfa_long.csv"
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")

long_df = add_visit_time(pd.read_csv(long_path), visit_col="visit")
pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
pair_long_df = trackfa_pairs_to_long(pairs_df)
feature_groups = infer_trackfa_feature_groups(pairs_df)
cat = feature_catalog(long_df, pairs_df)
imaging_cols = [c for c in feature_groups.all_neuroimaging if c in long_df.columns]
if not imaging_cols:
    imaging_cols = cat["long_imaging_columns"]

print(f"Loaded {long_path.name}: {long_df.shape[0]} rows, {long_df['subject_id'].nunique()} subjects")
print(f"Loaded {pairs_path.name}: {pairs_df.shape[0]} rows")
print(f"MRI features available for reporting: {len(imaging_cols)}")


Canonical modelling-cohort pair counts


,count,interval,n,definition
0,N12,V1->V2,108,subjects with a V1V2 annual pair row
1,N23,V2->V3,99,subjects with a V2V3 annual pair row
2,N13,V1->V3,90,subjects with both V1V2 and V2V3 annual pair rows
3,N123,"V1,V2,V3",90,subjects represented across all three visits v...


Loaded trackfa_long.csv: 522 rows, 174 subjects
Loaded trackfa_pairs_drop3poms.csv: 207 rows
MRI features available for reporting: 146


## Model Performance Table

The SRM Global Linear model is recomputed here on the subject-level longitudinal table for the primary and secondary intervals. Other model families are appended from existing optimization logs using the same required question/metric schema; unavailable metrics remain explicitly marked as unavailable rather than inferred.


In [2]:
pair_imaging_cols = [c for c in feature_groups.all_neuroimaging if c in pair_long_df.columns]
annual_pooled_result = srm_global_loocv(
    pair_long_df,
    pair_imaging_cols,
    subject_col="pair_id",
    visit_col="visit",
    selection_method="none",
    k=8,
    ridge=1e-6,
    covariance_shrinkage=0.45,
    z_clip=None,
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    compute_ci=False,
    split_group_col="subject",
    start_visit=1,
    end_visit=2,
)
annual_interval_summary = adjacent_pair_interval_effect_summary(
    annual_pooled_result["oof_df"],
    pair_col="pair_id",
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)

# Reconstruct V1->V3 from annual-pair OOF scores for subjects with both V1V2 and V2V3 rows.
oof = annual_pooled_result["oof_df"].copy()
parsed = oof["pair_id"].astype(str).str.extract(r"(?P<subject>.+)_(?P<pair_type>V1V2|V2V3)$")
oof = oof.assign(subject_id=parsed["subject"], pair_type=parsed["pair_type"])
v1_scores = (
    oof.loc[oof["pair_type"].eq("V1V2") & oof["visit"].eq(1), ["subject_id", "score"]]
    .rename(columns={"score": "score_v1"})
)
v3_scores = (
    oof.loc[oof["pair_type"].eq("V2V3") & oof["visit"].eq(2), ["subject_id", "score"]]
    .rename(columns={"score": "score_v3"})
)
v13_pairs = v1_scores.merge(v3_scores, on="subject_id", how="inner")
v13_long = pd.concat([
    v13_pairs[["subject_id", "score_v1"]].rename(columns={"score_v1": "score"}).assign(visit=1, time_years=0.0),
    v13_pairs[["subject_id", "score_v3"]].rename(columns={"score_v3": "score"}).assign(visit=3, time_years=2.0),
], ignore_index=True)
v13_summary = interval_effect_summary(
    v13_long,
    subject_col="subject_id",
    visit_col="visit",
    score_col="score",
    time_col="time_years",
    intervals=[(1, 3, "V1->V3", False)],
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)

pooled_annual_summary = pooled_adjacent_pair_effect_summary(
    annual_pooled_result["oof_df"],
    pair_col="pair_id",
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
composite_intervals = pd.concat([
    annual_interval_summary,
    v13_summary,
    pooled_annual_summary,
], ignore_index=True)
print("Composite interval performance from annual pair-table OOF predictions")
display(composite_intervals)
print("Counts are tied to trackfa_pairs_drop3poms.csv: N12=108, N23=99, N13=90, N123=90. Pooled annual d_z uses pair OOF predictions with split_group_col='subject'.")

srm_interval_results = {
    "V1->V2": annual_pooled_result,
    "V2->V3": annual_pooled_result,
    "V1->V3": {"oof_df": v13_long, "selected_features_by_fold": annual_pooled_result["selected_features_by_fold"]},
}


Composite interval performance from annual pair-table OOF predictions


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive,start_visit,end_visit,annualised
0,V1->V2,108,2.093881,1.872556,1.118194,0.924416,1.408201,0.898148,NaN,NaN,NaN
1,V2->V3,99,1.799046,2.387154,0.753636,0.560923,0.976512,0.767677,NaN,NaN,NaN
2,V1->V3,90,3.932395,2.755918,1.426891,1.245247,1.735865,0.944444,1,3,False
3,V1->V2 + V2->V3,207,1.952873,2.134023,0.915113,0.769638,1.101774,0.835749,NaN,NaN,NaN


Counts are tied to trackfa_pairs_drop3poms.csv: N12=108, N23=99, N13=90, N123=90. Pooled annual d_z uses pair OOF predictions with split_group_col='subject'.


In [3]:
single_feature_intervals = single_feature_interval_baselines(
    long_df,
    imaging_cols,
    subject_col="subject_id",
    visit_col="visit",
    time_col="time_years",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
print("Strongest individual MRI features by interval")
display(single_feature_intervals.groupby("interval", group_keys=False).head(5))

clinical_vars = [c for c in ["FARS", "SARA", "mfars_total", "sara_total"] if c in long_df.columns]
clinical_interval_parts = []
for clinical_col in clinical_vars:
    clinical_interval_parts.append(
        interval_effect_summary(
            long_df.dropna(subset=[clinical_col]),
            subject_col="subject_id",
            visit_col="visit",
            score_col=clinical_col,
            time_col="time_years",
            n_boot=N_BOOT,
            seed=RANDOM_SEED,
        ).assign(feature=clinical_col, kind="clinical_scale")
    )
clinical_intervals = pd.concat(clinical_interval_parts, ignore_index=True) if clinical_interval_parts else pd.DataFrame()
print("Clinical-scale benchmarks")
display(clinical_intervals)


Strongest individual MRI features by interval


,interval,start_visit,end_visit,annualised,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive,feature,kind
0,V1->V2,V1,V2,True,149,-1224.194644,1857.318129,-0.659120,-0.886112,-0.437390,0.214765,Cerebellum_Cortex_CerebNet,single_mri_feature
1,V1->V2,V1,V2,True,149,-1347.653107,2112.606208,-0.637910,-0.876610,-0.418949,0.214765,Cerebellum_CerebNet,single_mri_feature
2,V1->V2,V1,V2,True,149,-1535.215013,2437.834877,-0.629745,-0.879234,-0.411334,0.201342,cerebellumFS,single_mri_feature
3,V1->V2,V1,V2,True,149,-1251.871389,2023.449903,-0.618682,-0.859347,-0.399685,0.234899,Cerebellum_Cortex_FS,single_mri_feature
4,V1->V2,V1,V2,True,149,-191.253947,338.599147,-0.564839,-0.792716,-0.388927,0.268456,Whole_brainstem,single_mri_feature
146,V1->V3,V1,V3,False,134,-2795.656343,2862.038521,-0.976806,-1.167725,-0.831867,0.164179,cerebellumFS,single_mri_feature
147,V1->V3,V1,V3,False,134,-2192.912799,2250.074356,-0.974596,-1.147427,-0.811102,0.141791,Cerebellum_Cortex_FS,single_mri_feature
148,V1->V3,V1,V3,False,134,-2409.456649,2536.964178,-0.949740,-1.154683,-0.790027,0.164179,Cerebellum_CerebNet,single_mri_feature
149,V1->V3,V1,V3,False,134,-2091.460799,2218.296284,-0.942823,-1.133571,-0.794404,0.134328,Cerebellum_Cortex_CerebNet,single_mri_feature
150,V1->V3,V1,V3,False,135,-2362.170370,2916.727256,-0.809870,-1.050714,-0.644354,0.140741,Cereb_vol,single_mri_feature


Clinical-scale benchmarks


,interval,start_visit,end_visit,annualised,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive,feature,kind
0,V1->V3,V1,V3,False,150,5.271111,8.010399,0.658033,0.527607,0.802225,0.733333,mfars_total,clinical_scale
1,V1->V2,V1,V2,True,165,3.032323,5.592166,0.542245,0.407338,0.694389,0.660606,mfars_total,clinical_scale
2,V2->V3,V2,V3,True,149,2.204698,5.800426,0.380092,0.256318,0.556495,0.637584,mfars_total,clinical_scale
3,V1->V3,V1,V3,False,150,2.543333,3.491090,0.728521,0.566647,0.895133,0.733333,sara_total,clinical_scale
4,V1->V2,V1,V2,True,164,1.387195,2.604459,0.532623,0.411083,0.675898,0.658537,sara_total,clinical_scale
5,V2->V3,V2,V3,True,150,1.216667,2.983971,0.407734,0.246863,0.578295,0.620000,sara_total,clinical_scale


In [4]:
def clinical_annual_effects_from_pairs(pairs: pd.DataFrame, delta_cols: list[str]) -> pd.DataFrame:
    """Recalculate annual clinical benchmarks directly from paired annual deltas."""
    interval = pairs["patient_id"].astype(str).str.extract(r"_(V\dV\d)$")[0]
    rows = []
    interval_specs = [
        ("V1->V2", interval.eq("V1V2")),
        ("V2->V3", interval.eq("V2V3")),
        ("pooled annual", interval.isin(["V1V2", "V2V3"])),
    ]
    for col in delta_cols:
        clinical_name = col.removeprefix("delta_")
        for label, mask in interval_specs:
            values = pd.to_numeric(pairs.loc[mask, col], errors="coerce").dropna()
            sd = values.std(ddof=1)
            rows.append({
                "clinical_score": clinical_name,
                "interval": label,
                "n_pairs": int(values.shape[0]),
                "mean_delta": values.mean(),
                "sd_delta": sd,
                "d_z": values.mean() / sd if sd and not np.isnan(sd) else np.nan,
                "p_delta_gt_0": (values > 0).mean() if values.shape[0] else np.nan,
            })
    return pd.DataFrame(rows)

clinical_delta_cols = [
    c for c in ["delta_mfars_total", "delta_sara_total", "delta_adl_total"] if c in pairs_df.columns
]
clinical_annual_pair_benchmarks = clinical_annual_effects_from_pairs(pairs_df, clinical_delta_cols)
print("Clinical annual benchmarks recalculated directly from trackfa_pairs_drop3poms.csv")
display(clinical_annual_pair_benchmarks)

clinical_pooled_confirmation = (
    clinical_annual_pair_benchmarks
    .query("interval == 'pooled annual'")
    .loc[:, ["clinical_score", "n_pairs", "mean_delta", "sd_delta", "d_z", "p_delta_gt_0"]]
    .sort_values("clinical_score")
    .reset_index(drop=True)
)
print("Pooled annual clinical d_z confirmation")
display(clinical_pooled_confirmation)

for score in ["mfars_total", "sara_total"]:
    match = clinical_pooled_confirmation.loc[clinical_pooled_confirmation["clinical_score"].eq(score)]
    if not match.empty:
        row = match.iloc[0]
        print(
            f"Confirmed {score} pooled annual d_z = {row['d_z']:.3f} "
            f"(N={int(row['n_pairs'])}, P(delta>0)={row['p_delta_gt_0']:.3f})."
        )


Clinical annual benchmarks recalculated directly from trackfa_pairs_drop3poms.csv


,clinical_score,interval,n_pairs,mean_delta,sd_delta,d_z,p_delta_gt_0
0,mfars_total,V1->V2,108,2.294753,4.852690,0.472883,0.638889
1,mfars_total,V2->V3,99,1.681818,4.982457,0.337548,0.626263
2,mfars_total,pooled annual,207,2.001610,4.912805,0.407427,0.632850
3,sara_total,V1->V2,108,1.291667,2.424683,0.532716,0.648148
4,sara_total,V2->V3,99,0.828283,2.847449,0.290886,0.535354
5,sara_total,pooled annual,207,1.070048,2.639077,0.405463,0.594203
6,adl_total,V1->V2,108,1.013889,2.551762,0.397329,0.629630
7,adl_total,V2->V3,99,0.782828,2.871537,0.272617,0.505051
8,adl_total,pooled annual,207,0.903382,2.705234,0.333938,0.570048


Pooled annual clinical d_z confirmation


,clinical_score,n_pairs,mean_delta,sd_delta,d_z,p_delta_gt_0
0,adl_total,207,0.903382,2.705234,0.333938,0.570048
1,mfars_total,207,2.001610,4.912805,0.407427,0.632850
2,sara_total,207,1.070048,2.639077,0.405463,0.594203


Confirmed mfars_total pooled annual d_z = 0.407 (N=207, P(delta>0)=0.633).
Confirmed sara_total pooled annual d_z = 0.405 (N=207, P(delta>0)=0.594).


In [5]:
primary_oof = srm_interval_results["V1->V3"]["oof_df"]
clinical_validity_table = clinical_validity(
    primary_oof,
    long_df,
    subject_col="subject_id",
    visit_col="visit",
    score_col="score",
    clinical_variables=clinical_vars,
    start_visit="V1",
    end_visit="V3",
) if clinical_vars else pd.DataFrame()
print("Clinical validation of locked OOF composite scores")
display(clinical_validity_table)

stability_table = pd.DataFrame([
    {
        "model": "SRM Global Linear",
        "mean_jaccard": selected_feature_jaccard(
            [fs for result in srm_interval_results.values() for fs in result["selected_features_by_fold"]]
        )["mean_jaccard"],
        "sign_stability": np.nan,
        "score_ranking_stability": np.nan,
    }
])
print("Feature robustness diagnostics")
display(stability_table)


Clinical validation of locked OOF composite scores


,analysis,clinical_variable,rho,p_value,n
0,cross_sectional,mfars_total,0.362087,5.896292e-07,180
1,longitudinal_delta,mfars_total,0.037460,7.259428e-01,90
2,cross_sectional,sara_total,0.373039,2.504540e-07,180
3,longitudinal_delta,sara_total,0.106721,3.167530e-01,90


Feature robustness diagnostics


,model,mean_jaccard,sign_stability,score_ranking_stability
0,SRM Global Linear,1.0,NaN,NaN


In [6]:
performance = assemble_performance_rows(
    "SRM Global Linear",
    composite_intervals=composite_intervals,
    clinical_intervals=clinical_intervals,
    single_feature_intervals=single_feature_intervals,
    clinical_validity=clinical_validity_table,
    stability=stability_table,
    cv_mode=f"subject-level grouped {CV_N_SPLITS}-fold",
    source="model_performance.ipynb",
)

log_models = best_model_rows_from_logs([
    REPO_ROOT / "results" / "srm_composite_optimization_log.csv",
    REPO_ROOT / "results" / "comparator_optimization_log.csv",
    REPO_ROOT / "results" / "progression_dl_optimization_log.csv",
])
performance = append_log_model_summaries(performance, log_models)

performance_csv = save_one_performance_csv(performance, REPO_ROOT / "results" / "model_performance_summary.csv")
print(f"Saved one consolidated model-performance CSV: {performance_csv}")
display(performance)


Saved one consolidated model-performance CSV: /Users/robertwang/Documents/New_project/biomarkers/results/model_performance_summary.csv


,model,question,metric,role,value,n,status,evidence,cv_mode,source
0,SRM Global Linear,12-month sensitivity V1->V2,"V1->V2 paired d_z, CI, N, P(delta>0)",Primary,"1.1181938866675425 [0.9244156504576596, 1.4082...",108.0,computed,composite V1->V2 OOF annual interval,subject-level grouped 5-fold,model_performance.ipynb
1,SRM Global Linear,12-month sensitivity V2->V3,"V2->V3 paired d_z, CI, N, P(delta>0)",Primary temporal replication,"0.7536362848231071 [0.5609228107663216, 0.9765...",99.0,computed,composite V2->V3 OOF annual interval,subject-level grouped 5-fold,model_performance.ipynb
2,SRM Global Linear,12-month pooled annual sensitivity,"Pooled V1->V2 + V2->V3 paired d_z, CI, N, P(de...",Pooled annual diagnostic,"0.9151132057198257 [0.7696384922193471, 1.1017...",207.0,computed,pooled OOF annual pair deltas; participant-gro...,subject-level grouped 5-fold,model_performance.ipynb
3,SRM Global Linear,24-month cumulative sensitivity,V1->V3 paired d_z,Secondary,1.426891,90.0,computed,composite V1->V3 cumulative,subject-level grouped 5-fold,model_performance.ipynb
4,SRM Global Linear,Direction consistency,P(delta > 0),Secondary,0.8981481481481481; 0.7676767676767676,108.0,computed,annual V1->V2 and V2->V3 P(delta>0),subject-level grouped 5-fold,model_performance.ipynb
...,...,...,...,...,...,...,...,...,...,...
103,FusionMLP_group_kfold,Better than MRI alone?,vs strongest individual MRI feature,RQ1,NaN,NaN,not_available_in_existing_log,clinical heads disabled; tuned/reported on mea...,group_kfold,progression_dl_optimization_log.csv
104,FusionMLP_group_kfold,Disease specific?,FRDA vs control change,Specificity,NaN,NaN,not_available_in_existing_log,clinical heads disabled; tuned/reported on mea...,group_kfold,progression_dl_optimization_log.csv
105,FusionMLP_group_kfold,Clinically meaningful?,Spearman Z vs FARS/SARA,RQ3,NaN,NaN,not_available_in_existing_log,clinical heads disabled; tuned/reported on mea...,group_kfold,progression_dl_optimization_log.csv
106,FusionMLP_group_kfold,Tracks clinical change?,Spearman delta Z vs delta FARS/SARA,Strong RQ3,NaN,NaN,not_available_in_existing_log,clinical heads disabled; tuned/reported on mea...,group_kfold,progression_dl_optimization_log.csv
